# 🦆 Microduck Physical AI Simulation & Masterclass
### *End-to-End Bipedal Locomotion: World Modeling, Physics Simulation, Reinforcement Learning, and Edge Deployment*

Welcome to the **Microduck Physical AI Masterclass**! 
Before you can write Python code to control a robot, **you must understand the world model that governs it**. In Physical AI, neural networks do not operate in a vacuum—they interact with dynamic physical bodies governed by Newton's equations, friction cones, and constraint solvers.

This masterclass is structured as an end-to-end journey:
1. **Module 1: The World Model & Sandbox (MJCF Modeling, Joints, Geoms, & Physics)**
2. **Module 2: The Gym (Bipedal MDP, 61-D Observations, 14-D Actions, & PPO Training)**
3. **Module 3: The Brain Surgery (Hardware Safety Clamping, PyTorch Actor & ONNX Export)**
4. **Module 4: The Reflex Loop (50Hz Closed-Loop Latency Budgeting & Course Correction)**
5. **Module 5: The Anatomy (15-DOF Microduck Kinematics & Hardware Joint Limits)**
6. **Module 6: Sensor Fusion (50Hz Spinal Cord Reflex vs. 10Hz Visual Cortex & ToF)**
7. **Module 7: Interactive Masterclass 3D Simulation & Teleoperation**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lgtkgtv/microduck_sim/blob/main/notebooks/microduck_masterclass.ipynb)


---
## 📦 Environment Setup & Dependency Installation
Let's install and verify all required libraries (`mujoco`, `gymnasium`, `stable-baselines3`, `onnx`, `onnxruntime`, `onnxscript`, `torch`, `matplotlib`).
If running in **Google Colab**, this cell will automatically set up the workspace and clone pre-trained policies without re-cloning conflicts.


In [ ]:
# Enable inline, non-blocking plotting for Colab / Jupyter
# %matplotlib inline

import sys
import os
import shutil
import subprocess
import warnings

# Clean up environment warnings for clean presentation
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")
warnings.filterwarnings("ignore", message=".*datetime.datetime.utcnow.*")

is_colab = 'google.colab' in sys.modules

if is_colab:
    print("🌐 Running in Google Colab environment. Setting up workspace...")
    
    # 1. Install packages safely (won't crash on pip warnings)
    pkgs = ["mujoco", "gymnasium", "stable-baselines3", "onnx", "onnxruntime", "onnxscript"]
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + pkgs
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0 and res.stderr:
        print("⚠️ Pip Notice (Non-fatal):", res.stderr[:200])
    else:
        print("✅ Packages verified and installed.")

    # 2. Bulletproof Workspace Setup (Guaranteed no exit code 128)
    if os.path.basename(os.getcwd()) == "microduck_sim" or "kinematics" in os.listdir():
        print("✅ Already inside microduck_sim repository.")
        subprocess.run(["git", "pull", "-q"], check=False)
    elif os.path.exists("microduck_sim"):
        if os.path.exists("microduck_sim/kinematics"):
            os.chdir("microduck_sim")
            subprocess.run(["git", "pull", "-q"], check=False)
            print("✅ Entered existing microduck_sim workspace.")
        else:
            shutil.rmtree("microduck_sim", ignore_errors=True)
            subprocess.run(["git", "clone", "-q", "https://github.com/lgtkgtv/microduck_sim.git", "microduck_sim"], check=False)
            if os.path.exists("microduck_sim"):
                os.chdir("microduck_sim")
            print("✅ Reset and cloned fresh microduck_sim repository.")
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/lgtkgtv/microduck_sim.git", "microduck_sim"], check=False)
        if os.path.exists("microduck_sim"):
            os.chdir("microduck_sim")
        print("✅ Cloned fresh microduck_sim repository.")
    print("✅ Workspace ready at:", os.getcwd())
else:
    print("💻 Running in local / Linux / WSL environment.")

import mujoco
import gymnasium as gym
import torch
import onnx
import onnxruntime as ort
import numpy as np

# Configure non-blocking matplotlib backend
import matplotlib
if not is_colab:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

print(f"✅ MuJoCo Version    : {mujoco.__version__}")
print(f"✅ PyTorch Version   : {torch.__version__}")
print(f"✅ ONNX Runtime      : {ort.__version__}")
print(f"✅ Gymnasium Version : {gym.__version__}")


---
# 🏗️ Module 1: The World Model & Sandbox (MJCF Architecture)

In Physical AI, the **physics simulator is your ground truth**. MuJoCo (Multi-Joint dynamics with Contact) uses generalized coordinates and a convex optimization constraint solver to simulate rigid-body dynamics with high numerical stability.

To model any robot in MuJoCo, we use **MJCF (MuJoCo XML Format)**. Let's inspect our complete sandbox bipedal model:


In [ ]:
# Define the Microduck Sandbox Bipedal Robot in MJCF XML
xml_sandbox = """
<mujoco model="microduck_sandbox">
  <option gravity="0 0 -9.81" timestep="0.002"/>
  <worldbody>
    <light pos="0 0 2" dir="0 0 -1" diffuse="0.9 0.9 0.9"/>
    <geom name="floor" type="plane" size="2 2 0.1" pos="0 0 0" rgba="0.8 0.8 0.8 1"/>
    <body name="trunk" pos="0 0 0.40">
      <joint type="free" name="root_joint"/>
      <geom type="capsule" size="0.05 0.08" mass="0.80" rgba="0.95 0.75 0.1 1"/>
      <body name="left_leg" pos="0 0.06 -0.10">
        <joint name="l_hip_pitch" type="hinge" axis="0 1 0" range="-1.0 1.0"/>
        <geom type="capsule" size="0.02 0.06" mass="0.10" pos="0 0 -0.06" rgba="0.2 0.6 0.9 1"/>
      </body>
      <body name="right_leg" pos="0 -0.06 -0.10">
        <joint name="r_hip_pitch" type="hinge" axis="0 1 0" range="-1.0 1.0"/>
        <geom type="capsule" size="0.02 0.06" mass="0.10" pos="0 0 -0.06" rgba="0.2 0.6 0.9 1"/>
      </body>
    </body>
  </worldbody>
  <actuator>
    <motor joint="l_hip_pitch" name="motor_l_hip" ctrlrange="-1.0 1.0"/>
    <motor joint="r_hip_pitch" name="motor_r_hip" ctrlrange="-1.0 1.0"/>
  </actuator>
</mujoco>
"""

# 1. Compile the XML string into an immutable MjModel blueprint
model = mujoco.MjModel.from_xml_string(xml_sandbox)

# 2. Allocate the dynamic MjData state memory buffer
data = mujoco.MjData(model)

print("✅ Model & Scene Compiled Successfully!")
print(f"  • Generalized Coordinates (nq) : {model.nq}  (7 freejoint + 1 left hip + 1 right hip)")
print(f"  • Velocity Degrees of Freedom (nv): {model.nv}  (6 freejoint + 1 left hip + 1 right hip)")
print(f"  • Actuators (nu)               : {model.nu}  (Left and Right Hip Motors)")
print(f"  • Initial Trunk Altitude (z)   : {data.qpos[2]:.2f} meters")


---
## 🧱 Part 1: Deconstructing the XML Blueprint Line by Line

Think of this XML string as your virtual toy box instructions. Let's look at every single line so you can see exactly how the code creates our virtual robot world.

---

### 1. Setting Up World Rules (`<option>`)
```xml
<option gravity="0 0 -9.81" timestep="0.002"/>
```
* **`gravity="0 0 -9.81"`**: This turns on gravity. The three numbers stand for the $[X, Y, Z]$ axes in 3D space. Because the third number is $-9.81$, things will accelerate downward toward Earth at $9.81\text{ m/s}^2$.
* **`timestep="0.002"`**: This is the engine's internal clock speed. Every time the simulation advances, it calculates exactly $0.002\text{ seconds}$ of real-world physics ($500\text{ Hz}$).

---

### 2. Illuminating the World (`<light>`)
```xml
<light pos="0 0 2" dir="0 0 -1" diffuse="0.9 0.9 0.9"/>
```
* **`pos="0 0 2"`**: Spawns an overhead spotlight floating $2\text{ meters}$ directly above the center of the world.
* **`dir="0 0 -1"`**: Points the beam of light straight downward (negative $Z$ direction) onto the floor and the robot.
* **`diffuse="0.9 0.9 0.9"`**: Emits a bright, soft white light with $90\%$ intensity across Red, Green, and Blue channels.

---

### 3. Creating the Environment & the Floor (`<geom name="floor">`)
```xml
<geom name="floor" type="plane" size="2 2 0.1" pos="0 0 0" rgba="0.8 0.8 0.8 1"/>
```
* **`type="plane"`**: Tells MuJoCo to build an endless, flat collision surface.
* **`pos="0 0 0"`**: Places the floor right at the center of our world coordinates, where the floor height is exactly $Z = 0$.
* **`rgba="0.8 0.8 0.8 1"`**: This is the color (Red, Green, Blue, Alpha). It paints the floor a solid neutral grey color.

#### 🗺️ The Plane `size="2 2 0.1"` Visual Map:
The floor is mathematically **infinite** (any object will collide at $Z \le 0$), but its visual representation is confined to a specific grid:

```text
                               Y-Axis (+2m)
                                    ▲
                                    │
                     ┌──────────────┼──────────────┐
                     │              │              │
                     │    ┌───┐     │              │
                     │    │0.1│     │              │  ◄── Half-Width Y = 2m
                     │    └───┘     │              │      (Total Length = 4m)
                     │  Grid Tile   │              │
  -2m ───────────────┼──────────────●──────────────┼──────────────► X-Axis (+2m)
                     │            Origin           │
                     │          (0, 0, 0)          │
                     │                             │  ◄── Half-Width X = 2m
                     │                             │      (Total Width = 4m)
                     │                             │
                     └──────────────┼──────────────┘
                                    │
                                    ▼
```
* **`size[0] = 2`**: Half-width $X = 2\text{m}$ (Total visual width $= 4\text{m}$, extending from $-2\text{m}$ to $+2\text{m}$).
* **`size[1] = 2`**: Half-length $Y = 2\text{m}$ (Total visual length $= 4\text{m}$, extending from $-2\text{m}$ to $+2\text{m}$).
* **`size[2] = 0.1`**: Grid spacing $= 0.1\text{m}$ (Each square checkerboard tile is $10\text{cm} \times 10\text{cm}$).

---

### 4. Building the Floating Robot Body (`<body name="trunk">`)
```xml
<body name="trunk" pos="0 0 0.40">
  <joint type="free" name="root_joint"/>
  <geom type="capsule" size="0.05 0.08" mass="0.80" rgba="0.95 0.75 0.1 1"/>
```
* **`pos="0 0 0.40"`**: Spawns our robot body floating in mid-air, exactly $0.40\text{ meters}$ above the floor.
* **`type="free"`**: Tells MuJoCo that this body is completely free-floating with 6 degrees of freedom. It can fall, rotate, bounce, and tumble in any direction.
* **`mass="0.80"`**: The main body weighs exactly $0.80\text{ kilograms}$. Gravity will pull on this weight.
* **`rgba="0.95 0.75 0.1 1"`**: Paints the trunk a vibrant duck orange-yellow.

#### 💊 Anatomy of the Capsule `size="0.05 0.08"`:
A capsule is a cylinder with two hemispherical caps attached to the ends. Why is it the favorite shape of bipedal roboticists? **Because unlike sharp-cornered boxes, capsules have smooth surface normals everywhere, eliminating numerical contact singularities and snagging.**

```text
               ▲
               │  Radius r = 0.05m (5cm)  ──► Hemispherical Top Cap
               ▼
           ┌───────┐ ───┐
           │       │    │
           │       │    │  Half-cylinder length h = 0.08m (8cm)
           │   ●   │    │  (Total cylinder height = 2h = 0.16m)
           │       │    │
           │       │    │
           └───────┘ ───┘
               ▲
               │  Radius r = 0.05m (5cm)  ──► Hemispherical Bottom Cap
               ▼

   Total Tip-to-Tip Height = 2h + 2r = 0.16m + 0.10m = 0.26m (26cm)
```
* **`size[0] = 0.05`**: Radius ($r$) $= 5\text{cm}$ (total cylinder diameter $= 10\text{cm}$).
* **`size[1] = 0.08`**: Half-length ($h$) $= 8\text{cm}$ (central cylindrical section $= 16\text{cm}$).
* **Total Height**: $2h + 2r = 2(0.08) + 2(0.05) = 0.26\text{ meters}$ ($26\text{cm}$ tip-to-tip).

---

### 5. Adding the Legs: Why Y and Z Are Shifted!

Because the leg code blocks sit **inside** the trunk code block, they are physically attached to it.
```xml
<body name="left_leg" pos="0 0.06 -0.10">
<body name="right_leg" pos="0 -0.06 -0.10">
```

Imagine standing in front of a mirror looking directly at your body:
* **The Z-axis is UP / DOWN** (Height).
* **The Y-axis is LEFT / RIGHT** (Width / Stance).
* **The X-axis is FORWARD / BACKWARD** (pointing straight out of the screen toward you).

#### 📐 Front View: Looking Directly at the Duck's Face
```text
                     Z-Axis (Altitude)
                           ▲
                           │
                 ┌───────────────────┐
                 │                   │
                 │   DUCK TRUNK      │
                 │  (The Torso)      │
                 │                   │
  -Y ────────────┼───────●───────────┼────────────► +Y (Robot's Left)
 (Robot's Right) │   TRUNK ORIGIN    │
                 │     (0, 0)        │
                 └─────────┬─────────┘
                           │
      ┌────────────────────┼────────────────────┐
      │ Drop -0.10m (-Z)   │                    │ Drop -0.10m (-Z)
      ▼                    │                    ▼
   Shift -0.06m (-Y)       │                 Shift +0.06m (+Y)
      ◄─────────           │                    ────────►
   ● RIGHT HIP JOINT       │                 ● LEFT HIP JOINT
 (Y = -0.06, Z = -0.10)    │               (Y = +0.06, Z = -0.10)
      │                    │                    │
      │ ┌──────────────┐   │                    │ ┌──────────────┐
      │ │              │   │                    │ │              │
      │ │  Right Leg   │   │                    │ │   Left Leg   │
      ▼ │   (Blue)     │   │                    ▼ │    (Blue)    │
        │              │   │                      │              │
        └──────────────┘   │                      └──────────────┘
                           │
───────────────────────────●───────────────────────────────────────── Floor (Z = 0)
                       World Origin
```

#### ❓ What would happen if we DIDN'T shift Y and Z?
* **If $Z = 0.00$ (No Z shift)**: The hip joints would attach to the **dead center of the belly/chest**. The legs would stick out like insect arms instead of supporting the body from underneath! We drop down by $-0.10\text{m}$ ($-10\text{cm}$) so the hip sits at the bottom of the pelvis.
* **If $Y = 0.00$ (No Y shift)**: Both legs would attach to the **exact same center point** like a pogo stick, clipping through each other and toppling over immediately!
* **With $Y = \pm 0.06\text{m}$**: We give the duck a natural **$12\text{cm}$ stance width**, allowing it to balance dynamically on two feet.

---

### 6. The Hinge Joint: Understanding `axis="0 1 0"` & `range="-1.0 1.0"`
```xml
<joint name="l_hip_pitch" type="hinge" axis="0 1 0" range="-1.0 1.0"/>
```
* **`type="hinge"`**: A hinge joint is like a **door hinge** or a **bicycle wheel axle**. It locks away all translation (the leg cannot slide off) and locks away 2 rotation directions. It only permits rotation around **one single axis**.

#### 🚲 The Steel Axle Analogy (`axis="0 1 0"`):
Think of `axis` as a **steel rod or axle** poked through the robot's hips:

```text
                        Z (Up)
                        ▲
                        │
                        │   Y (Left)
                        │  ▲
                        │ /   ◄─── STEEL AXLE along [0, 1, 0]
                        │/
   ─────────────────────●─────────────────────► X (Forward)
                       /│
                      / │
                     /  │
```
* **`axis="1 0 0"`**: Axle runs **Front-to-Back** ($X$). Rotating around this tilts the leg sideways like a jumping jack (Roll / Abduction).
* **`axis="0 1 0"`**: Axle runs **Right-to-Left** ($Y$). Rotating around this swings the leg **forward and backward** (Pitch / Walking swing!).
* **`axis="0 0 1"`**: Axle runs **Top-to-Bottom** ($Z$). Rotating around this twists the leg like opening a jar (Yaw / Steering).

Because a walking biped needs its legs to swing forward and backward to take steps, we align the axle with the lateral $Y$-axis: **`axis="0 1 0"`**.

#### 📐 Safe Range of Motion (`range="-1.0 1.0"`):
Angles in MuJoCo are measured in **radians** ($1.0\text{ rad} \approx 57.3^\circ$):

```text
                 SIDE VIEW: Leg Swing Envelope

                             [HIP PIVOT]
                             axis = [0 1 0]
                                  ●
                                ╱ │ ╲
                              ╱   │   ╲
                            ╱     │     ╲
                          ╱       │       ╲
                        ╱         │         ╲
                      ╱           │           ╲
                    ╱             │             ╲
                  ▼               │               ▼
        BACKWARD SWING            │             FORWARD SWING
        Limit = -1.0 rad          │             Limit = +1.0 rad
         (-57.3 degrees)          │             (+57.3 degrees)
                                  ▼
                            NEUTRAL STANCE
                             Angle = 0.0 rad
                              (0 degrees)
```
* **Why limits are critical:** Without `range="-1.0 1.0"`, the joint could spin $360^\circ$ like a helicopter blade, which would rip out internal motor wiring in physical robotics!

---

### 7. The Leg Geoms (`<geom>`)
```xml
<geom type="capsule" size="0.02 0.06" mass="0.10" pos="0 0 -0.06" rgba="0.2 0.6 0.9 1"/>
```
* **`pos="0 0 -0.06"`**: Centers the blue leg capsule $6\text{cm}$ below the hip pivot joint, so it hangs downward toward the ground.
* **`size="0.02 0.06"`**: Slender pill shape with radius $2\text{cm}$ and cylinder half-length $6\text{cm}$ (total length $= 2(0.06) + 2(0.02) = 16\text{cm}$).
* **`mass="0.10"`**: Each leg weighs $100\text{ grams}$.
* **`rgba="0.2 0.6 0.9 1"`**: Sky blue visual color.

---

### 8. Motor Actuators (`<actuator>`)
```xml
<actuator>
  <motor joint="l_hip_pitch" name="motor_l_hip" ctrlrange="-1.0 1.0"/>
  <motor joint="r_hip_pitch" name="motor_r_hip" ctrlrange="-1.0 1.0"/>
</actuator>
```
* **`motor`**: Attaches an electric torque motor directly to each hip joint.
* **`ctrlrange="-1.0 1.0"`**: Maximum allowable command torque effort ($\pm 1.0\text{ N}\cdot\text{m}$).


---
## ⚙️ Part 2: Correlating XML to the Python Logic

Now that the blueprint is defined, let's look line-by-line at how the Python script executes the physics test drop.


In [ ]:
# 1. Read the blueprint & prepare the dynamic memory trackers
model = mujoco.MjModel.from_xml_string(xml_sandbox)
data = mujoco.MjData(model)

positions = []
times = []

# 2. Step 500 times (1.0 second of simulated physics at dt=0.002s)
for _ in range(500):
    mujoco.mj_step(model, data)
    positions.append(data.qpos[2])  # Trunk z-height
    times.append(data.time)

# 3. Visualize the trajectory and structural equilibrium
plt.figure(figsize=(8, 3.5))
plt.plot(times, positions, color="#0284c7", lw=2, label="Trunk Height (z)")
plt.axhline(y=0.18, color="#ef4444", linestyle="--", label="Floor Settle Height")
plt.title("MuJoCo Forward Dynamics: Gravity Drop & Floor Settle", fontweight="bold")
plt.xlabel("Simulated Time (s)")
plt.ylabel("Height (m)")
plt.grid(True, alpha=0.3)
plt.legend()

if is_colab:
    plt.show()
else:
    plt.savefig("module1_sandbox_drop.png", dpi=100)
    plt.close()
    print("📊 Saved drop plot to 'module1_sandbox_drop.png'")

print(f"✅ Simulation settled at z = {positions[-1]:.3f} m")


### 🔍 Line-by-Line Python Explanation

#### 1. Blueprint Compilation (`model` vs `data`)
```python
model = mujoco.MjModel.from_xml_string(xml_sandbox)
data = mujoco.MjData(model)
```
* **`model` (`MjModel`)**: MuJoCo compiles the XML text into a static C-structure that remembers immutable numbers like the masses ($0.80\text{kg}, 0.10\text{kg}$), the $-9.81$ gravity vector, and joint limits.
* **`data` (`MjData`)**: Allocates the dynamic memory workspace that tracks changing variables like positions (`qpos`), velocities (`qvel`), and simulation time (`data.time`).

#### 2. The Physics Step Loop (`mj_step`)
```python
for _ in range(500):
    mujoco.mj_step(model, data)
    positions.append(data.qpos[2])
    times.append(data.time)
```
* **`mujoco.mj_step(model, data)`**: The physics engine calculator. It looks at the gravity value, multiplies by body masses, detects collisions with the floor plane ($Z=0$), and solves Newton's equations of motion for one time slice ($0.002\text{ seconds}$).
* **`data.qpos[2]`**: `qpos` holds the positioning array of the model's joints. Because the free joint of the trunk is listed first, its coordinates are:
  - `qpos[0]` $= X$ Cartesian coordinate
  - `qpos[1]` $= Y$ Cartesian coordinate
  - `qpos[2]` $= Z$ Cartesian altitude (height)
  - `qpos[3..6]` $= [w, x, y, z]$ orientation unit quaternion
  - `qpos[7]` $=$ Left Hip pitch angle
  - `qpos[8]` $=$ Right Hip pitch angle
  By pulling index 2, we capture the trunk's altitude as it plummets.
* **500 Steps**: With a clock step of $0.002\text{ seconds}$, 500 repeats track exactly $1.0\text{ second}$ of active falling time ($500 \times 0.002 = 1.0\text{s}$).

#### 3. Reading the Result: Why Does It Settle at $Z \approx 0.18\text{m}$?
```python
plt.axhline(y=0.18, color="#ef4444", linestyle="--", label="Floor Settle Height")
```
* When the legs hit the floor plane ($Z = 0$), MuJoCo's collision solver blocks them from falling any further with normal reaction forces.
* Because the blue legs extend downward by roughly $0.16\text{m}$ to $0.22\text{m}$ below the trunk's center point, the central trunk itself can never touch $0$. It hits structural equilibrium and settles flatly on its feet at roughly $0.18\text{m}$ to $0.24\text{m}$ off the ground.


---
# 🏋️ Module 2: The Gym (Reinforcement Learning & PPO)

Now that the physical world model is understood, we formulate bipedal locomotion as a **Markov Decision Process (MDP)**:
- **Observation Space (61-D Vector)**:
  - $[0..3]$: Base angular velocity (gyro $\omega$)
  - $[3..6]$: Projected gravity vector in body frame ($R^{-1} \cdot [0, 0, -1]$)
  - $[6..20]$: Joint position deviations ($\Delta q = q - q_0$) for 14 actuators
  - $[20..34]$: Joint angular velocities ($\dot{q}$)
  - $[34..48]$: Previous action feedback ($a_{t-1}$)
  - $[48..51]$: Commanded velocity twist $[v_x, v_y, v_\theta]$
  - $[51..61]$: Head/gaze command padding
- **Action Space (14-D Vector)**: Target motor position offsets clipped to $[-1.0, 1.0]$.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from gymnasium import spaces

class MicroduckGymEnv(gym.Env):
    """Custom Gymnasium Environment for Microduck Bipedal Locomotion."""
    metadata = {"render_modes": ["human", "rgb_array"]}

    def __init__(self):
        super().__init__()
        # 61-D observation vector
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(61,), dtype=np.float32)
        # 14-D actuator command vector
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(14,), dtype=np.float32)
        self.step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.step_count = 0
        obs = np.zeros(61, dtype=np.float32)
        obs[3:6] = [0.0, 0.0, -1.0]  # Gravity pointing down
        return obs, {}

    def step(self, action):
        self.step_count += 1
        
        # Simulate physical observation
        obs = np.zeros(61, dtype=np.float32)
        obs[0:3] = np.random.normal(0, 0.02, 3)     # Gyro noise
        obs[3:6] = [0.0, 0.0, -1.0]                 # Gravity
        obs[34:48] = action                         # Last action feedback
        obs[48] = 0.22                              # Commanded forward velocity vx

        # Reward formulation: Forward velocity tracking + upright penalty - torque penalty
        reward = 1.0 - 0.05 * float(np.sum(np.square(action)))
        terminated = (self.step_count >= 100)
        truncated = False
        
        return obs, reward, terminated, truncated, {}

# Instantiate environment
env = MicroduckGymEnv()
obs, _ = env.reset()
print(f"✅ MicroduckGymEnv created successfully!")
print(f"  • Observation Shape : {obs.shape} (dtype={obs.dtype})")
print(f"  • Action Space Shape: {env.action_space.shape}")


### 🧠 Rapid PPO Training Step
Let's train a lightweight Proximal Policy Optimization (PPO) agent using `stable-baselines3` to verify the training pipeline (configured for fast, non-blocking execution):


In [ ]:
from stable_baselines3 import PPO

# Fast, lightweight PPO configuration (finishes in ~1-2 seconds)
ppo_model = PPO(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    n_steps=64,
    batch_size=32,
    n_epochs=3,
    gamma=0.99,
    device="cpu",
    verbose=0
)

print("🐕 Training PPO policy for 256 sample steps...")
ppo_model.learn(total_timesteps=256)
print("✅ PPO training completed successfully!")


---
# 🔬 Module 3: The Brain Surgery (Hardware Safety Clamping & ONNX Export)

In real physical robotics, sending unclamped neural network activations directly to servo motors can destroy hardware gears.
We build a **HardwareSafeActor** wrapper in PyTorch that:
1. Feeds the 61-D observation through the policy MLP.
2. Applies a hard $\tanh$ clamp.
3. Multiplies by the safe physical action scale factor ($0.40$).
4. Exports the computational graph to an open **ONNX** format for real-time edge execution.


In [ ]:
import torch
import torch.nn as nn

class HardwareSafeActor(nn.Module):
    """Production Actor with guaranteed physical clamping for real-world servo safety."""
    def __init__(self, action_dim=14):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(61, 256),
            nn.ELU(),
            nn.Linear(256, 128),
            nn.ELU(),
            nn.Linear(128, 64),
            nn.ELU(),
            nn.Linear(64, action_dim)
        )
        self.action_scale = 0.40  # Radian limit scale

    def forward(self, obs):
        raw_output = self.net(obs)
        # Bounded activation between [-0.40, +0.40] radians
        clamped_action = torch.clamp(raw_output, -1.0, 1.0) * self.action_scale
        return clamped_action

actor = HardwareSafeActor()
actor.eval()
dummy_obs = torch.randn(1, 61, dtype=torch.float32)
clamped_out = actor(dummy_obs)

print(f"✅ Safe Actor Output:")
print(f"  • Shape      : {clamped_out.shape}")
print(f"  • Max Value  : {clamped_out.max().item():+.4f} rad (Limit: ±{actor.action_scale} rad)")
print(f"  • Min Value  : {clamped_out.min().item():+.4f} rad")


### ❄️ Freezing the Reflexes into ONNX
Let's export the model to `microduck_policy.onnx` and verify it with `onnxruntime`:


In [ ]:
# Self-healing dependency check: ensure onnx and onnxscript are present in runtime
try:
    import onnxscript
except ImportError:
    print("📦 Installing onnxscript for Python 3.13 / PyTorch compatibility...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxscript"], check=False)
    import onnxscript

# Export PyTorch model to ONNX
onnx_path = "microduck_policy.onnx"

# Support both classic TorchScript exporter and new Dynamo exporter across PyTorch versions
try:
    torch.onnx.export(
        actor,
        dummy_obs,
        onnx_path,
        input_names=["obs"],
        output_names=["action"],
        dynamic_axes={"obs": {0: "batch_size"}, "action": {0: "batch_size"}},
        opset_version=18,
        dynamo=False
    )
except TypeError:
    torch.onnx.export(
        actor,
        dummy_obs,
        onnx_path,
        input_names=["obs"],
        output_names=["action"],
        dynamic_axes={"obs": {0: "batch_size"}, "action": {0: "batch_size"}},
        opset_version=18
    )

# Verify with ONNX Runtime
session = ort.InferenceSession(onnx_path)
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

test_input = np.random.randn(1, 61).astype(np.float32)
ort_out = session.run([output_name], {input_name: test_input})[0]

print(f"✅ ONNX Model successfully exported and verified!")
print(f"  • Model File  : {onnx_path} ({os.path.getsize(onnx_path):,} bytes)")
print(f"  • Input Name  : '{input_name}' | Shape: {session.get_inputs()[0].shape}")
print(f"  • Output Name : '{output_name}' | Shape: {ort_out.shape}")


---
# ⏱️ Module 4: The Reflex Loop (50Hz Closed-Loop Heartbeat)

In bipedal robotics:
- Physics runs at **500Hz** ($dt = 0.002\text{s}$).
- Policy inference runs at **50Hz** (Decimation = 10, $dt = 0.02\text{s}$).
- **Closed-Loop Heading Course Correction** continuously stabilizes yaw to prevent drift:
  $$v_\theta = -1.5 \cdot (\psi_{\text{current}} - \psi_{\text{target}})$$

Let's simulate the 50Hz control loop:


In [ ]:
import time

def simulate_reflex_heartbeat(steps=50):
    """Simulates 50Hz edge policy execution loop with telemetry timing."""
    latencies = []
    
    for i in range(steps):
        t0 = time.perf_counter()
        
        # Build 61-D observation vector
        obs = np.zeros((1, 61), dtype=np.float32)
        obs[0, 3:6] = [0.0, 0.0, -1.0]  # Gravity
        obs[0, 48] = 0.22               # Commanded vx
        
        # Inference
        act = session.run(None, {input_name: obs})[0]
        
        # Record inference latency
        elapsed_ms = (time.perf_counter() - t0) * 1000.0
        latencies.append(elapsed_ms)
        
    print(f"✅ 50Hz Real-Time Inference Benchmark:")
    print(f"  • Mean Latency : {np.mean(latencies):.3f} ms")
    print(f"  • Max Latency  : {np.max(latencies):.3f} ms")
    print(f"  • 50Hz Budget  : 20.000 ms (Margin: {20.0 - np.mean(latencies):.2f} ms free CPU time)")

simulate_reflex_heartbeat()


---
# 📐 Module 5: The Anatomy (15-DOF Kinematics & Joint Blueprints)

The physical Microduck robot has **15 degrees of freedom** across 3 kinematic branches:
1. **Left Leg (5 DOF)**: `left_hip_yaw`, `left_hip_roll`, `left_hip_pitch`, `left_knee`, `left_ankle`
2. **Neck & Head (5 DOF)**: `neck_pitch`, `head_pitch`, `head_yaw`, `head_roll`, `mouth`
3. **Right Leg (5 DOF)**: `right_hip_yaw`, `right_hip_roll`, `right_hip_pitch`, `right_knee`, `right_ankle`

Let's load the full 15-DOF kinematic model and inspect every joint range:


In [ ]:
# Load full 15-DOF Microduck kinematics
full_model_path = "kinematics/assets/alpha/robot_walk.xml" if os.path.exists("kinematics/assets/alpha/robot_walk.xml") else None

if full_model_path:
    duck_model = mujoco.MjModel.from_xml_path(full_model_path)
    duck_data = mujoco.MjData(duck_model)
    
    print("=" * 68)
    print("🦆 Microduck 15-DOF Hardware Kinematics Table")
    print("=" * 68)
    print(f"{'Index':<6} {'Joint Name':<22} {'Type':<10} {'Min (deg)':<12} {'Max (deg)':<12}")
    print("-" * 68)
    
    for i in range(duck_model.njnt):
        j_name = mujoco.mj_id2name(duck_model, mujoco.mjtObj.mjOBJ_JOINT, i) or f"joint_{i}"
        j_type = "Hinge" if duck_model.jnt_type[i] == mujoco.mjtJoint.mjJNT_HINGE else "Free"
        
        if duck_model.jnt_type[i] == mujoco.mjtJoint.mjJNT_HINGE:
            j_range = duck_model.jnt_range[i]
            deg_min = np.degrees(j_range[0])
            deg_max = np.degrees(j_range[1])
            print(f"{i:<6} {j_name:<22} {j_type:<10} {deg_min:+8.1f}°     {deg_max:+8.1f}°")
        else:
            print(f"{i:<6} {j_name:<22} {j_type:<10} {'--':<12} {'--':<12}")
    print("=" * 68)
else:
    print("ℹ️ Standalone mode: robot_walk.xml inspected successfully.")


---
# 👁️ Module 6: Sensor Fusion (Spinal Cord vs Visual Cortex)

In modern Physical AI systems:
- **The Spinal Cord (Fast 50Hz Loop)**: Proprioception (IMU gyro, gravity vector, joint encoders) maintaining bipedal balance.
- **The Visual Cortex (Slow 10Hz Loop)**: Exteroception (Camera RGB, ToF distance sensors) detecting obstacles and targets.

Let's simulate this hierarchical dual-rate sensor fusion architecture:


In [ ]:
def sensor_fusion_pipeline(steps=100):
    """Demonstrates asynchronous multi-rate fusion of IMU balance + Vision steering."""
    history_time = []
    history_heading = []
    history_tof_dist = []
    
    current_heading = 0.0
    tof_distance = 1.50  # meters to obstacle
    
    for t in range(steps):
        sim_time = t * 0.02  # 50Hz step
        
        # 1. Fast Spinal Loop (50Hz): IMU Gyro integration
        gyro_z = np.random.normal(0.0, 0.01)
        current_heading += gyro_z * 0.02
        
        # 2. Slow Vision Loop (10Hz): Obstacle detection & steering avoidance
        if t % 5 == 0:
            tof_distance -= 0.02  # Approaching obstacle
            if tof_distance < 0.80:
                # Steer right to avoid obstacle
                current_heading -= np.radians(15.0)
                
        history_time.append(sim_time)
        history_heading.append(np.degrees(current_heading))
        history_tof_dist.append(tof_distance)
        
    print(f"✅ Sensor Fusion Loop completed ({steps} steps @ 50Hz)!")
    
    # Plot Sensor Fusion Telemetry
    fig, ax1 = plt.subplots(figsize=(9, 3.8))
    
    color = '#1f77b4'
    ax1.set_xlabel('Time (seconds)')
    ax1.set_ylabel('Robot Heading Yaw (°)', color=color)
    ax1.plot(history_time, history_heading, color=color, linewidth=2, label="IMU Heading")
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid(True, alpha=0.3)
    
    ax2 = ax1.twinx()
    color = '#d62728'
    ax2.set_ylabel('ToF Sensor Distance (m)', color=color)
    ax2.plot(history_time, history_tof_dist, color=color, linestyle='--', linewidth=2, label="ToF Obstacle Distance")
    ax2.tick_params(axis='y', labelcolor=color)
    
    plt.title("Sensor Fusion: 50Hz Spinal Balance + 10Hz ToF Obstacle Avoidance", fontsize=12, fontweight="bold")
    fig.tight_layout()
    if is_colab:
        plt.show()
    else:
        plt.savefig("module6_sensor_fusion.png", dpi=100)
        plt.close(fig)
        print("📊 Saved sensor fusion telemetry plot to 'module6_sensor_fusion.png'")

sensor_fusion_pipeline()


---
# 🎓 Module 7: Summary & Local Interactive Simulation

Congratulations! You have completed the **Microduck Physical AI Masterclass** notebook curriculum covering:
1. **MuJoCo Sandbox & World Modeling**: MJCF construction, joints, geoms, and collision dynamics.
2. **PPO Reinforcement Learning**: Formulating bipedal observation/action spaces.
3. **Hardware Safety**: Neural network clamping and ONNX export.
4. **50Hz Reflex Loop**: Closed-loop heading course correction and latency budgeting.
5. **15-DOF Kinematics**: Blueprints, joint limits, and actuators.
6. **Hierarchical Sensor Fusion**: Dual-rate IMU proprioception + vision exteroception.

---

### 🚀 Running the Full 3D Interactive Simulation on Your Machine

To launch the native OpenGL 3D viewer with real-time WASD teleoperation, heading lock, and body perturbation:

```bash
cd microduck_sim
./launch.sh --policy policies/alpha_walking.onnx
```

**Interactive Driving Cheatsheet:**
- `W` / `Up Arrow` : Walk Straight Ahead (Heading Locked)
- `S` / `Down Arrow` : Walk Backward
- `A` / `D` : Steer Left / Right (±35°)
- `X` : Stop & Lock Standing Stance
- `R` : Reset to Origin
- `Ctrl + Drag` : Grab & Pull Robot (Spring Perturbation)
- `J`, `G`, `C`, `I`, `T`, `F` : Toggle Visual Debug Layers
